## Forecaster: XGBoost
The linear weighted composite from `forecaster.ipynb` fails on forward labels: test AUC barely
clears 0.5 and sits below the equal-weight baseline across most alpha values. The relationship
between indicators and 60-day-ahead drawdowns is nonlinear and interactions matter.

This notebook skips weight re-optimization and goes directly to XGBoost with
`TimeSeriesSplit(gap=LOOKAHEAD)`. Metrics: AUC and Brier score. Baseline: equal-weight stress score.

In [ ]:
import os
from itertools import product
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.metrics import brier_score_loss, roc_auc_score, roc_curve
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier

N_JOBS = os.cpu_count()
print(f"using {N_JOBS} cores")

### 1. Load data

In [ ]:
try:
    df = pd.read_parquet("data/processed/stress_score.parquet")
    DOCS_PATH = Path("docs")
except FileNotFoundError:
    df = pd.read_parquet("../data/processed/stress_score.parquet")
    DOCS_PATH = Path("../docs")

df.index = pd.to_datetime(df.index)

INDICATOR_COLS = [
    "T10Y2Y",
    "T10Y3M",
    "T30Y10Y",
    "USALOLITOAASTSAM",
    "UMCSENT",
    "PERMIT",
    "NEWORDER",
    "ICSA",
    "DRCCLACBS",
    "BAMLH0A0HYM2",
    "XLK_XLV",
    "TLT",
    "HG=F",
    "CL=F",
    "EEM",
    "DX=F",
]

df = df.dropna(subset=INDICATOR_COLS)
print(f"{len(df)} rows, {df.index[0].date()} to {df.index[-1].date()}")

### 2. Forward drawdown labels

In [ ]:
DRAWDOWN_THRESHOLD = 0.10
LOOKAHEAD = 60

spy = df["SPY"]

future_min = spy[::-1].rolling(window=LOOKAHEAD, min_periods=LOOKAHEAD).min()[::-1].shift(-1)
future_drop = future_min / spy - 1

labels_fwd = pd.Series(np.nan, index=spy.index)
valid = future_min.notna()
labels_fwd[valid] = (future_drop[valid] <= -DRAWDOWN_THRESHOLD).astype(float)
labels_fwd = labels_fwd.dropna().astype(int)

X_data = df.loc[labels_fwd.index, INDICATOR_COLS]
y_data = labels_fwd

n_pos = y_data.sum()
n_neg = len(y_data) - n_pos
scale_pos_weight = n_neg / n_pos

print(f"{len(y_data)} rows; {n_pos} forward drawdown events ({y_data.mean():.1%})")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

### 3. Baseline (equal-weight stress score)

In [ ]:
N = len(INDICATOR_COLS)
equal_score = X_data.sum(axis=1) / N

tscv_base = TimeSeriesSplit(n_splits=5, gap=LOOKAHEAD)
baseline_aucs, baseline_briers = [], []

for _, test_idx in tscv_base.split(X_data):
    y_test = y_data.iloc[test_idx]
    eq_test = equal_score.iloc[test_idx]
    if y_test.nunique() < 2:
        continue
    baseline_aucs.append(roc_auc_score(y_test, eq_test))
    baseline_briers.append(brier_score_loss(y_test, eq_test))

print(f"equal-weight baseline  AUC: {np.mean(baseline_aucs):.4f}  Brier: {np.mean(baseline_briers):.4f}")

### 4. XGBoost CV (base params)
Shallow trees (`max_depth=3`) and a low learning rate reduce overfitting on a dataset where
drawdown events are sparse and the signal is noisy. `scale_pos_weight` corrects for class imbalance.

In [ ]:
def run_xgb_cv(params, X, y, n_splits=5, gap=LOOKAHEAD):
    tscv = TimeSeriesSplit(n_splits=n_splits, gap=gap)
    folds = []

    for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        if y_train.nunique() < 2 or y_test.nunique() < 2:
            continue

        model = XGBClassifier(**params, eval_metric="logloss", verbosity=0)
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]

        folds.append(
            {
                "fold": fold + 1,
                "test_start": X.index[test_idx[0]].date(),
                "test_end": X.index[test_idx[-1]].date(),
                "auc": roc_auc_score(y_test, proba),
                "brier": brier_score_loss(y_test, proba),
            }
        )

    return folds


BASE_PARAMS = {
    "n_estimators": 200,
    "max_depth": 3,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "scale_pos_weight": scale_pos_weight,
    "random_state": 42,
    "tree_method": "hist",
}

base_folds = run_xgb_cv(BASE_PARAMS, X_data, y_data)

print("XGBoost base CV:")
for f in base_folds:
    print(f"  fold {f['fold']} ({f['test_start']} to {f['test_end']}):  AUC={f['auc']:.4f}  Brier={f['brier']:.4f}")
print()
print(f"mean AUC:   {np.mean([f['auc'] for f in base_folds]):.4f}")
print(f"mean Brier: {np.mean([f['brier'] for f in base_folds]):.4f}")

### 5. Hyperparameter sweep

In [ ]:
PARAM_GRID = {
    "n_estimators": [100, 200, 500],
    "max_depth": [3, 4, 5],
    "learning_rate": [0.01, 0.05, 0.1],
}

param_combos = [
    {"n_estimators": n, "max_depth": d, "learning_rate": lr}
    for n, d, lr in product(PARAM_GRID["n_estimators"], PARAM_GRID["max_depth"], PARAM_GRID["learning_rate"])
]

print(f"{len(param_combos)} combinations")


def _sweep_job(combo, X, y):
    params = {
        **combo,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "scale_pos_weight": n_neg / n_pos,
        "random_state": 42,
        "tree_method": "hist",
    }
    folds = run_xgb_cv(params, X, y)
    return {
        **combo,
        "mean_auc": np.mean([f["auc"] for f in folds]),
        "mean_brier": np.mean([f["brier"] for f in folds]),
        "std_auc": np.std([f["auc"] for f in folds]),
    }


sweep_raw = Parallel(n_jobs=N_JOBS)(delayed(_sweep_job)(c, X_data, y_data) for c in param_combos)
sweep_results = sorted(sweep_raw, key=lambda r: r["mean_auc"], reverse=True)
print("sweep complete")

In [ ]:
print(f"{'n_est':>6}  {'depth':>5}  {'lr':>5}  {'AUC':>7}  {'std':>6}  {'Brier':>7}")
for r in sweep_results[:15]:
    print(
        f"{r['n_estimators']:>6}  {r['max_depth']:>5}  {r['learning_rate']:>5}  "
        f"{r['mean_auc']:>7.4f}  {r['std_auc']:>6.4f}  {r['mean_brier']:>7.4f}"
    )

In [ ]:
# penalize variance: prefer stable AUC over lucky single-fold peaks
best = max(sweep_results, key=lambda r: r["mean_auc"] - r["std_auc"])
print(
    f"best: n_estimators={best['n_estimators']}  max_depth={best['max_depth']}  "
    f"learning_rate={best['learning_rate']}\n"
    f"      AUC={best['mean_auc']:.4f} +/- {best['std_auc']:.4f}  Brier={best['mean_brier']:.4f}"
)

### 6. Final CV with best params

In [ ]:
BEST_PARAMS = {
    "n_estimators": best["n_estimators"],
    "max_depth": best["max_depth"],
    "learning_rate": best["learning_rate"],
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "scale_pos_weight": scale_pos_weight,
    "random_state": 42,
    "tree_method": "hist",
}

best_folds = run_xgb_cv(BEST_PARAMS, X_data, y_data)

print("final CV (best params):")
for f in best_folds:
    print(f"  fold {f['fold']} ({f['test_start']} to {f['test_end']}):  AUC={f['auc']:.4f}  Brier={f['brier']:.4f}")
print()
print(f"baseline           AUC: {np.mean(baseline_aucs):.4f}  Brier: {np.mean(baseline_briers):.4f}")
print(f"XGBoost (tuned)    AUC: {np.mean([f['auc'] for f in best_folds]):.4f}  Brier: {np.mean([f['brier'] for f in best_folds]):.4f}")

### 7. Feature importance

In [ ]:
# fit on full dataset for feature importance — in-sample only
final_model = XGBClassifier(**BEST_PARAMS, eval_metric="logloss", verbosity=0)
final_model.fit(X_data, y_data)

importances = final_model.feature_importances_
order = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(np.array(INDICATOR_COLS)[order], importances[order], color="#1f77b4", alpha=0.7)
ax.set_title("XGBoost feature importance (gain, full dataset)")
ax.set_xlabel("importance")
plt.tight_layout()
plt.show()

### 8. Probability visualization

In [ ]:
drawdown_prob = pd.Series(final_model.predict_proba(X_data)[:, 1], index=X_data.index)

STRESS_PERIODS = [
    ("2007-10-01", "2009-03-01", "GFC"),
    ("2011-07-01", "2011-10-01", "EU debt crisis"),
    ("2015-08-01", "2016-02-01", "China slowdown"),
    ("2018-10-01", "2018-12-31", "Q4 selloff"),
    ("2020-02-01", "2020-04-01", "COVID crash"),
    ("2022-01-01", "2022-10-01", "Rate hike cycle"),
    ("2025-01-20", "2025-04-01", "Tariff shock"),
]

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

ax1.fill_between(drawdown_prob.index, drawdown_prob, alpha=0.35, color="#d62728", label="drawdown prob")
ax1.plot(drawdown_prob.index, drawdown_prob, color="#d62728", linewidth=0.6)
ax2.plot(df.index, df["SPY"], color="#1f77b4", linewidth=1.0, label="SPY")

for start, end, label in STRESS_PERIODS:
    ax1.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.08, color="grey")
    mid = pd.Timestamp(start) + (pd.Timestamp(end) - pd.Timestamp(start)) / 2
    ax1.text(
        mid, 0.97, label, ha="center", va="top", fontsize=7.5,
        color="#444444", transform=ax1.get_xaxis_transform(),
    )

ax1.set_ylabel("probability", color="#d62728")
ax2.set_ylabel("SPY", color="#1f77b4")
ax1.set_ylim(0, 1)
ax1.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax1.yaxis.grid(True, linewidth=0.3, color="grey", alpha=0.25)
ax1.set_axisbelow(True)
ax2.yaxis.grid(False)
ax1.tick_params(axis="y", colors="#d62728")
ax2.tick_params(axis="y", colors="#1f77b4")
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax1.xaxis.set_major_locator(mdates.YearLocator())

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3)

fig.suptitle("XGBoost Drawdown Probability vs SPY", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

for score, label, color in [
    (equal_score, f"equal weight (AUC {roc_auc_score(y_data, equal_score):.3f})", "#aec7e8"),
    (drawdown_prob, f"XGBoost (AUC {roc_auc_score(y_data, drawdown_prob):.3f})", "#d62728"),
]:
    fpr, tpr, _ = roc_curve(y_data, score)
    ax.plot(fpr, tpr, label=label)

ax.plot([0, 1], [0, 1], color="grey", linewidth=0.8, linestyle="--")
ax.set_xlabel("false positive rate")
ax.set_ylabel("true positive rate")
ax.set_title("ROC curves vs forward labels (full data, in-sample)")
ax.legend()
plt.tight_layout()
plt.show()